In [ ]:
!pip install transformers[torch] datasets accelerate scikit-learn pandas numpy

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import Dataset

# Verify GPU is detected
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# 1. Load Data
data_path = 'AUGMENTED_BALANCED_DATASET.csv'
df = pd.read_csv(data_path)

# Label Encoding
le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])
num_labels = len(le.classes_)
joblib.dump(le, 'label_encoder.pkl')

# Stratified split 80/20
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Use datasets
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
eval_dataset = Dataset.from_pandas(eval_df[['text', 'label']])

In [ ]:
# 2. Tokenize Data
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [ ]:
# 3. Model Definition & Metrics
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# 4. Training Configuration (ITERATION 2)
training_args = TrainingArguments(
    output_dir='./results_iter2',
    num_train_epochs=5,                  # INCREASED: Train for 5 epochs instead of 3
    learning_rate=1e-5,                  # DECREASED: Slightly smaller learning rate for finer tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=1000,                   # INCREASED: Longer warmup for stability
    weight_decay=0.02,                   # INCREASED: Double weight decay prevents overfitting
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Stop if it doesn't improve for 2 epochs
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
# 5. Run Training
print("\n--- Starting Iteration 2 Fine-Tuning ---")
trainer.train()


--- Starting Iteration 2 Fine-Tuning ---


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.804645,0.730559,0.738400,0.738270,0.758387,0.738400
2,0.596341,0.548586,0.798000,0.797706,0.807170,0.798000
3,0.450577,0.520399,0.816800,0.816383,0.823757,0.816800
4,0.323217,0.486482,0.836000,0.835950,0.843690,0.836000
5,0.245474,0.485654,0.840400,0.840523,0.847825,0.840400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3125, training_loss=0.5852713626098632, metrics={'train_runtime': 1352.4536, 'train_samples_per_second': 36.97, 'train_steps_per_second': 2.311, 'total_flos': 3288976780800000.0, 'train_loss': 0.5852713626098632, 'epoch': 5.0})

In [ ]:
# 6. Final Evaluation
print("\n==================================")
print("        FINAL EVALUATION          ")
print("==================================")
metrics = trainer.evaluate()
print(metrics)

predictions = trainer.predict(tokenized_eval)
preds = np.argmax(predictions.predictions, axis=1)
print("\nClassification Report:")
print(classification_report(eval_df['label'], preds, target_names=le.classes_))


        FINAL EVALUATION          


{'eval_loss': 0.48565438389778137, 'eval_accuracy': 0.8404, 'eval_f1': 0.840523228700318, 'eval_precision': 0.8478250993318328, 'eval_recall': 0.8404, 'eval_runtime': 17.7304, 'eval_samples_per_second': 141.001, 'eval_steps_per_second': 4.456, 'epoch': 5.0}

Classification Report:
              precision    recall  f1-score   support

     Account       0.91      0.76      0.83       500
     Billing       0.80      0.78      0.79       500
    Delivery       0.91      0.85      0.88       500
     Product       0.75      0.93      0.83       500
     Service       0.87      0.89      0.88       500

    accuracy                           0.84      2500
   macro avg       0.85      0.84      0.84      2500
weighted avg       0.85      0.84      0.84      2500



In [ ]:
# 7. Save Best Model
output_dir = './best_model_iter2'
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

import shutil
from google.colab import files

shutil.make_archive('best_model_iter2', 'zip', 'best_model_iter2')
print("Created best_model_iter2.zip for download.")
files.download('best_model_iter2.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Created best_model_iter2.zip for download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>